<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/03_Quantization_basics_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Quantization Basics 실습**

**실습 목표**
1. FP32 tensor를 INT8 범위로 양자화하고 다시 복원합니다.
2. symmetric / asymmetric quantization의 차이를 확인합니다.
3. activation quantization에서 calibration이 왜 필요한지 확인합니다.
4. 간단한 모델에 weight quantization과 activation quantization을 적용합니다.

# 0. Colab 환경설정
- colab에서 GPU를 사용할 수 있도록 세팅
    - 런타임 > 런타임 유형 변경 > Python 3 와 T4 GPU 선택
- colab에서 Google Drive에 접근할 수 있도록 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("[현재 파일 위치]")
!pwd
print("[현재 디렉토리의 파일 확인]")
!ls

Mounted at /content/drive
[현재 파일 위치]
/content
[현재 디렉토리의 파일 확인]
drive  sample_data


**day 2** 폴더로 이동해주세요.

왼쪽의 **폴더** 아이콘을 클릭하면 경로를 쉽게 볼 수 있습니다.

In [2]:
# 본인 환경에 맞게 경로를 수정하여 사용하세요.
%cd /content/drive/MyDrive/day2
!pwd

/content/drive/.shortcut-targets-by-id/1-vYRZOLYLvUD5Kma-yELIfM0X7TGxy15/day2
/content/drive/.shortcut-targets-by-id/1-vYRZOLYLvUD5Kma-yELIfM0X7TGxy15/day2


# 1. Weight Quantization

이번 절에서는 모델의 weight tensor를 직접 quantization 합니다.

- Weight quantization은 이미 학습된 모델의 파라미터를 더 낮은 bit-width로 표현하는 과정입니다.
- 여기서는 실제 INT8 kernel을 사용하는 것이 아니라, `quantize -> dequantize` 과정을 직접 구현하여 quantization error를 확인합니다.
- 이후 모델의 weight에 동일한 과정을 적용합니다.

## 1-1. Absolute Maximum (Absmax)

- Absmax 양자화에서는 원래 숫자를 텐서의 절대 최대값으로 나누고, 입력값을 [-127, 127] 범위에 매핑하기 위해 스케일링 팩터(127)를 곱합니다.
- 원래의 FP32 값을 복원하려면 INT8 숫자를 양자화 계수로 나누며, 이 과정에서 반올림으로 인한 일부 정밀도 손실이 발생할 수 있습니다.

$$
X_{quant} = round(\frac{127}{max(|X|)} * X)
$$

$$
X_{dequant} = \frac{X_{quant}}{scale}
$$

위의 수식에서 $127 / max(|X|)$ 는 스케일에 해당합니다.

In [4]:
# 코드에서는 PyTorch를 이용하여 입력된 텐서 X를 절대 최대값을 기준으로 양자화(quantization)하는 함수를 정의합니다.

# torch library 을 import
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
import random
import numpy as np
import os
import time
from copy import deepcopy
from utils import *


def set_random_seed(seed):
    # 고정된 시드를 설정하여 결과의 재현성을 보장하기 위한 함수
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(1)


def absmax_quantize(X):
    """
    절대 최대값을 기준으로 양자화(Quantization)하는 함수
    Args:
        X (torch.Tensor): 입력 텐서
    Returns:
        (torch.Tensor, torch.Tensor, torch.Tensor): 양자화된 텐서, 복원된 텐서, scale
    """

    ###### [실습] `torch.max`를 사용하여 `torch.abs(X)`의 최댓값을 구하고 scale을 계산하세요. ######
    scale = 127 / torch.max(torch.abs(X))

    # Quantize
    X_quant = torch.clip((scale * X).round(), -128, 127)

    ###### [실습] Dequantize: scale로 나누어 복원합니다.
    X_dequant = X_quant / scale

    # quantization 된 값과, dequantization 된 값을 반환
    return X_quant, X_dequant, scale

## 1-2. Zero-point Quantization

- 제로 포인트 양자화를 사용하면 비대칭 입력 분포를 고려할 수 있습니다.
- 예를 들어 ReLU 함수의 출력처럼 양수 값이 많은 activation을 다룰 때 유용합니다.
- 입력 값은 먼저 값의 전체 범위(255)를 최대값과 최소값의 차이로 나눈 값으로 스케일링됩니다.
- 그런 다음 이 분포를 제로 포인트로 이동시켜 [-128, 127] 범위에 매핑합니다.
    - Absmax와 비교했을 때 `zero_point`가 추가되는 것을 주목합니다.

$$
scale={255 \over max(X)-min(X)}
$$

$$
zeropoint=-round(scale*min(X))-128
$$

- 다음으로 weight 또는 activation을 quantization 하거나 dequantization 할 수 있습니다.

$$
X_{quant}=round(scale*X+zeropoint)
$$

$$
X_{dequant}={X_{quant}-zeropoint \over scale}
$$

In [6]:
def zeropoint_quantize(X):
    """
    제로포인트를 사용하여 양자화하는 함수
    Args:
        X (torch.Tensor): 입력 텐서
    Returns:
        (torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor): 양자화된 텐서, 복원된 텐서, zeropoint, scale
    """

    # 값 범위 계산 (분모)
    x_range = torch.max(X) - torch.min(X)  # 최대값과 최소값의 차이로 값 범위 계산
    x_range = 1 if x_range == 0 else x_range  # 값 범위가 0인 경우 1로 설정하여 division by zero 방지

    ###### [실습] 8-bit signed quantization의 전체 범위인 255를 사용하여 scale을 계산하세요. ######
    scale = 255 / x_range  # 최대 범위 255로 매핑하는 스케일 계산 (8비트 양자화 기준)

    # 제로포인트 계산
    zeropoint = (-scale * torch.min(X) - 128).round()  # 최소값을 기준으로 제로포인트 설정 후 반올림

    # 양자화 수행
    ###### [실습] 양자화된 값을 -128에서 127 사이로 클리핑하세요. ######
    X_quant = torch.clip((X * scale + zeropoint).round(), -128, 127)

    # 복원 수행 (역양자화)
    ###### [실습] 양자화된 값을 제로포인트로 보정한 후 스케일로 나누어 복원하세요. ######
    X_dequant = (X_quant - zeropoint) / scale

    return X_quant, X_dequant, zeropoint, scale

In [7]:
# [실습] -10 ~ 100 사이의 랜덤한 값 10개를 가지는 텐서 생성
random_x = torch.randint(-10, 100, (10,)).float()

# [실습] absmax_quantize 함수를 호출하여 quantization 수행
abs_x, abs_x_dequant, abs_scale = absmax_quantize(random_x)

# [실습] zeropoint_quantize 함수를 호출하여 quantization 수행
zero_x, zero_x_dequant, zero_zp, zero_scale = zeropoint_quantize(random_x)

In [8]:
# 텐서의 최솟값과 최댓값을 출력합니다. 이는 양자화하기 전의 원본 텐서 범위를 보여줌
print(f"Random X, Min: {random_x.min()}, Max: {random_x.max()}")

# 절대 최대값 양자화를 수행한 abs_x 텐서의 최소값과 최대값을 출력
print(f"Absmax Quantized X, Min: {abs_x.min()}, Max: {abs_x.max()}")
print(f"Absmax Dequantization MSE: {torch.mean((random_x - abs_x_dequant) ** 2):.6f}")

# 제로포인트 양자화를 수행한 zero_x 텐서의 최소값과 최대값을 출력
print(f"Zeropoint Quantized X, Min: {zero_x.min()}, Max: {zero_x.max()}")
print(f"Zeropoint Dequantization MSE: {torch.mean((random_x - zero_x_dequant) ** 2):.6f}")

Random X, Min: -9.0, Max: 94.0
Absmax Quantized X, Min: -12.0, Max: 127.0
Absmax Dequantization MSE: 0.053264
Zeropoint Quantized X, Min: -128.0, Max: 127.0
Zeropoint Dequantization MSE: 0.011168


# 2. Activation Quantization

실제 inference에서는 weight뿐 아니라 activation도 함께 양자화되는 경우가 많습니다.

- Weight는 학습이 끝나면 고정됩니다.
- Activation은 입력 데이터마다 값의 범위가 달라집니다.
- 따라서 activation quantization에서는 대표 데이터로 activation range를 미리 관찰하는 `calibration` 과정이 필요합니다.

아래 실습에서는 activation tensor를 직접 양자화하고, outlier가 있을 때 quantization error가 어떻게 커지는지 확인합니다.

## 2-1. Activation Tensor 양자화

activation은 보통 layer의 출력입니다.

ReLU 이후 activation은 0 이상의 값을 많이 가지므로 asymmetric quantization을 사용하는 경우가 많습니다.

아래 함수는 관찰된 최솟값과 최댓값을 사용하여 activation을 INT8 범위로 양자화한 뒤 다시 FP32로 복원합니다.

앞에서 구현한 `zeropoint_quantize()`는 입력 tensor 자체의 min/max를 사용했습니다.

반면 activation quantization에서는 calibration 단계에서 미리 관찰한 `x_min`, `x_max`를 사용할 수 있어야 합니다.

따라서 아래 함수는 `x_min`, `x_max`가 주어지면 해당 range를 기준으로 quantization을 수행하고, 주어지지 않으면 현재 tensor의 min/max를 사용합니다.

In [9]:
def quantize_dequantize_asymmetric(X, x_min=None, x_max=None):
    """
    주어진 range를 사용하여 asymmetric quantization 후 dequantization을 수행하는 함수
    Args:
        X (torch.Tensor): 입력 activation 또는 weight
        x_min: calibration으로 얻은 최솟값. None이면 현재 X의 최솟값 사용
        x_max: calibration으로 얻은 최댓값. None이면 현재 X의 최댓값 사용
    Returns:
        (torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor): quantized, dequantized, zeropoint, scale
    """
    if x_min is None:
        x_min = torch.min(X)
    else:
        x_min = torch.as_tensor(x_min, dtype=X.dtype, device=X.device)

    if x_max is None:
        x_max = torch.max(X)
    else:
        x_max = torch.as_tensor(x_max, dtype=X.dtype, device=X.device)

    x_range = x_max - x_min
    x_range = torch.tensor(1.0, device=X.device) if x_range == 0 else x_range

    ###### [실습] activation range를 8-bit signed quantization의 전체 범위인 255에 매핑하기 위한 scale을 계산하세요. ######
    scale = 255 / x_range
    ###### [실습] x_min이 -128에 대응되도록 zeropoint를 계산하세요. ######
    zeropoint = (-scale * x_min - 128).round()

    ###### [실습] scale과 zeropoint를 사용하여 activation X를 quantization 하세요. ######
    X_quant = torch.clip((X * scale + zeropoint).round(), -128, 127)
    ###### [실습] quantized activation에서 zeropoint를 빼고 scale로 나누어 다시 FP32 값으로 dequantization 하세요. ######
    X_dequant = (X_quant - zeropoint) / scale

    return X_quant, X_dequant, zeropoint, scale

# ReLU 이후의 activation을 가정한 예제 tensor
activation = torch.randn(1000).abs() * 2
act_quant, act_dequant, act_zp, act_scale = quantize_dequantize_asymmetric(activation)

print(f"Activation Min: {activation.min():.4f}, Max: {activation.max():.4f}")
print(f"Activation Quantized Min: {act_quant.min()}, Max: {act_quant.max()}")
print(f"Activation Dequantization MSE: {torch.mean((activation - act_dequant) ** 2):.6f}")
print(f"Scale: {act_scale:.4f}, Zeropoint: {act_zp:.4f}")

Activation Min: 0.0012, Max: 7.8910
Activation Quantized Min: -128.0, Max: 127.0
Activation Dequantization MSE: 0.000079
Scale: 32.3202, Zeropoint: -128.0000


## 2-2. Outlier가 Activation Quantization에 미치는 영향

activation range는 최솟값과 최댓값에 의해 결정됩니다.

따라서 매우 큰 outlier가 하나만 있어도 대부분의 값이 좁은 구간에 몰리게 되고, 실제로 중요한 값들의 quantization resolution이 나빠질 수 있습니다.

In [10]:
# outlier가 없는 activation
activation_normal = torch.randn(1000).abs() * 2
_, dequant_normal, _, _ = quantize_dequantize_asymmetric(activation_normal)
normal_mse = torch.mean((activation_normal - dequant_normal) ** 2)

# outlier가 있는 activation
activation_outlier = activation_normal.clone()
activation_outlier[0] = 50.0
_, dequant_outlier, _, _ = quantize_dequantize_asymmetric(activation_outlier)
outlier_mse = torch.mean((activation_outlier - dequant_outlier) ** 2)

print(f"MSE without outlier: {normal_mse:.6f}")
print(f"MSE with outlier   : {outlier_mse:.6f}")
print("\n[해석]")
print("outlier가 있으면 전체 activation range가 커지고, 일반적인 activation 값들의 표현 정밀도가 낮아질 수 있습니다.")

MSE without outlier: 0.000052
MSE with outlier   : 0.003179

[해석]
outlier가 있으면 전체 activation range가 커지고, 일반적인 activation 값들의 표현 정밀도가 낮아질 수 있습니다.


## 2-3. Calibration의 역할

Static quantization에서는 실제 모델을 변환하기 전에 일부 입력 데이터를 모델에 통과시켜 activation의 범위를 관찰합니다. 이 과정을 calibration이라고 합니다.

- calibration data가 너무 적으면 activation range를 잘못 추정할 수 있습니다.
- calibration data가 너무 많으면 시간이 오래 걸립니다.
- 일반적으로 실제 inference 입력과 비슷한 대표 데이터를 사용하는 것이 중요합니다.

In [12]:
# calibration data 개수에 따른 activation range 추정 차이를 확인합니다.
full_activation = torch.cat([
    torch.randn(900).abs() * 1.5,
    torch.randn(100).abs() * 5.0
])

def estimate_range(x, num_samples):
    sample = x[:num_samples]
    return sample.min(), sample.max()

###### [실습] calibration sample 수를 바꿔보며 estimated range와 MSE가 어떻게 변하는지 확인하세요. ######
for n in [10, 50, 100, 500, 1000, 5000, 10000]:
    x_min, x_max = estimate_range(full_activation, n)
    _, x_dequant, _, _ = quantize_dequantize_asymmetric(full_activation, x_min, x_max)
    mse = torch.mean((full_activation - x_dequant) ** 2)
    print(f"Calibration Samples: {n:4d} | Estimated Min: {x_min:7.4f} | Estimated Max: {x_max:7.4f} | MSE: {mse:.6f}")

Calibration Samples:   10 | Estimated Min:  0.1643 | Estimated Max:  2.0329 | MSE: 1.278674
Calibration Samples:   50 | Estimated Min:  0.0105 | Estimated Max:  3.3477 | MSE: 0.690040
Calibration Samples:  100 | Estimated Min:  0.0105 | Estimated Max:  3.7588 | MSE: 0.574480
Calibration Samples:  500 | Estimated Min:  0.0022 | Estimated Max:  4.3570 | MSE: 0.438879
Calibration Samples: 1000 | Estimated Min:  0.0022 | Estimated Max: 13.1599 | MSE: 0.000215
Calibration Samples: 5000 | Estimated Min:  0.0022 | Estimated Max: 13.1599 | MSE: 0.000215
Calibration Samples: 10000 | Estimated Min:  0.0022 | Estimated Max: 13.1599 | MSE: 0.000215


# 3. Demo with Simple Network (MNIST)

이번 절에서는 MNIST 분류 모델을 학습한 뒤, weight quantization과 activation quantization을 적용합니다.

수업 시간을 고려하여 전체 MNIST 중 일부만 사용합니다. 더 높은 정확도를 확인하고 싶다면 `USE_SMALL_SUBSET = False`로 변경하면 됩니다.

- get_size_of_model: 모델의 전체 크기를 측정합니다.
- benchmark_model: 모델의 평균 지연 시간(`baseline_latency`)과 이미지 한 장에 대한 추론 시간(`baseline_inference_time`)을 측정합니다.

In [13]:
# 1. 간단한 신경망 모델 정의
class SimpleNN(nn.Module):
    def __init__(self):
        # 부모 클래스 초기화
        super(SimpleNN, self).__init__()

        # 입력층: 28x28 이미지를 입력받아, 128개의 노드로 변환
        self.fc1 = nn.Linear(28*28, 128, bias=False)

        # 활성화 함수로 ReLU 사용
        self.relu = nn.ReLU()

        # 두 번째 은닉층: 128개의 노드를 입력받아 64개의 노드로 변환
        self.fc2 = nn.Linear(128, 64, bias=False)

       ###### [실습] 최종 출력층: 64개의 노드를 입력받아 10개의 클래스로 출력하는 레이어를 작성하세요. ######
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        # 순전파 정의
        # 이미지를 1차원 벡터로 변환
        x = x.view(-1, 28*28)

        # 첫 번째 층과 ReLU 활성화 적용
        x = self.relu(self.fc1(x))

        # 두 번째 층과 ReLU 활성화 적용
        x = self.relu(self.fc2(x))

        # 최종 출력층으로 이동
        x = self.fc3(x)

        # 최종 결과 반환
        return x


# 2. 데이터셋 로드 및 전처리
transform = transforms.Compose([

    # 이미지를 PyTorch 텐서로 변환
    transforms.ToTensor(),

    # MNIST 데이터셋의 평균과 표준 편차로 정규화 (채널 수에 맞추어 평균과 표준편차 지정)
    # MNIST 는 채널이 하나이기 때문에 하나의 값만 입력
    # 평균: 0.1307, 표준편차: 0.3081
    transforms.Normalize((0.1307,), (0.3081,))
])

# MNIST 데이터셋 로드 (훈련용)
train_dataset = datasets.MNIST(root='/content/data', train=True, transform=transform, download=True)

# MNIST 데이터셋 로드 (테스트용)
test_dataset = datasets.MNIST(root='/content/data', train=False, transform=transform, download=True)

# 데이터 로더 생성 (훈련 데이터)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 데이터 로더 생성 (테스트 데이터)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 3. 모델 학습 설정
# 사용할 장치 설정 (GPU가 사용 가능하면 GPU 사용)
# quantization 분석은 CPU에서도 충분히 수행 가능하지만, 학습은 GPU가 있으면 GPU를 사용합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 인스턴스 생성 후 해당 장치로 이동
model = SimpleNN().to(device)

# 손실 함수와 옵티마이저 설정
# 손실 함수로 Cross Entropy Loss 사용
criterion = nn.CrossEntropyLoss()

# 옵티마이저로 Adam 사용, 학습할 매개변수와 학습률 설정
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 모델 학습 함수 정의
def train_model(model, train_loader, criterion, optimizer, num_epochs=2):
    # 모델을 학습 모드로 설정
    model.train()

    # 주어진 에폭 수만큼 반복
    for epoch in range(num_epochs):

        # 현재 에폭의 손실 & 정확도 초기화
        running_loss = 0.0
        correct = 0
        total = 0

        # 데이터 로더에서 배치 단위로 이미지와 레이블 가져오기
        for images, labels in train_loader:
            # 데이터를 장치로 이동
            images, labels = images.to(device), labels.to(device)

            # 순전파
            outputs = model(images) # 모델을 통해 출력 계산
            loss = criterion(outputs, labels) # 손실 계산

            # 역전파 및 최적화
            optimizer.zero_grad() # 기울기 초기화
            loss.backward() # 역전파 수행
            optimizer.step() # 가중치 업데이트

            running_loss += loss.item() # 현재 배치의 손실 추가
            _, predicted = torch.max(outputs.data, 1) # 예측된 클래스 인덱스 계산
            total += labels.size(0) # 전체 샘플 수 업데이트
            correct += (predicted == labels).sum().item() # 올바르게 예측된 샘플 수 업데이트

        # 에폭 당 평균 손실과 정확도 출력
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Train Acc: {100 * correct / total:.2f}%')

# 5. 모델 테스트 함수 정의
def test_model(model, test_loader, print_result=True):
    # 모델을 평가 모드로 설정
    model.eval()
    correct = 0 # 맞춘 예측 수 초기화
    total = 0 # 전체 예측 수 초기화

    # 기울기 계산을 하지 않도록 설정
    with torch.no_grad():
        # 테스트 데이터 로딩
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device) # 데이터를 장치로 이동
            outputs = model(images) # 모델을 통해 출력 계산
            _, predicted = torch.max(outputs.data, 1) # 예측값 중 가장 큰 값의 인덱스 반환
            total += labels.size(0) # 전체 레이블 수 추가
            correct += (predicted == labels).sum().item() # 맞춘 예측 수 추가

    acc = 100 * correct / total
    if print_result:
        # 테스트 정확도 출력
        print(f'Test Accuracy: {acc:.2f}%')
    return acc

# 6. 모델 학습 및 테스트 실행
train_model(model, train_loader, criterion, optimizer, num_epochs=5) # 모델 학습
baseline_acc = test_model(model, test_loader) # 모델 테스트
baseline_size = get_size_of_model(model) # 모델 크기 측정
baseline_latency, baseline_inference_time = benchmark_model(model, test_loader, device) # 모델 추론 속도 측정
print(f"Model Size: {baseline_size:.2f} KB")
print(f"Average Latency per Batch: {baseline_latency * 1000:.4f} ms")
print(f"Average Inference Time per Image: {baseline_inference_time * 1000:.4f} ms")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.38MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 127kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.20MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.57MB/s]


Epoch [1/5], Loss: 0.2653, Train Acc: 92.14%
Epoch [2/5], Loss: 0.1124, Train Acc: 96.62%
Epoch [3/5], Loss: 0.0770, Train Acc: 97.58%
Epoch [4/5], Loss: 0.0603, Train Acc: 98.03%
Epoch [5/5], Loss: 0.0493, Train Acc: 98.42%
Test Accuracy: 97.23%
Model Size: 438.79 KB
Average Latency per Batch: 0.4143 ms
Average Inference Time per Image: 0.0004 ms


# 4. Quantize Models

이번 절에서는 위에서 구현한 `absmax_quantize`, `zeropoint_quantize` 함수를 사용하여 모델 weight를 양자화합니다.

주의할 점은 다음과 같습니다.

- 먼저 quantized weight 값을 그대로 모델에 저장했을 때 정확도가 어떻게 변하는지 확인합니다.
- Absmax 방식은 0을 중심으로 하는 대칭적인 scale 변환이므로, 이번처럼 bias가 없는 단순 FC 모델에서는 quantized 값을 그대로 넣어도 출력의 상대적인 구조가 비교적 유지되어 성능 저하가 작게 나타날 수 있습니다. 다만 모든 모델에서 항상 보장되는 결과는 아닙니다.
- 반면 Zero-point 방식은 scale과 zero-point offset이 함께 들어가는 비대칭 양자화이므로, quantized 값을 그대로 weight처럼 사용하면 원래 weight 분포와 크게 달라질 수 있습니다.
- 이후 Zero-point Quantization에서 dequantization 또는 scale/zero-point 보정이 왜 필요한지 확인합니다.


In [15]:
# 원래 모델의 가중치를 저장
weights = [param.data.clone() for param in model.parameters()]
# - `model.parameters()`는 모델의 모든 파라미터에 접근할 수 있게 해줍니다.
# - `param.data.clone()`는 각 파라미터 데이터를 복사하여, 원본 가중치 `weights` 리스트에 저장합니다.
#   양자화 과정에서 원본이 손상되지 않도록 복제합니다.

# 절대 최대값 양자화를 적용할 모델 생성
###### [실습] - `deepcopy`를 사용하여 원본 모델을 완전히 복사합니다. ######
model_abs = deepcopy(model)
# - 이 복사본(`model_abs`)에 절대 최대값 기반 양자화를 적용할 것입니다.

# 모든 가중치에 절대 최대값 양자화 적용
weights_abs = []
for param in model_abs.parameters():
    quantized, dequantized, scale = absmax_quantize(param.data)
    param.data = quantized.float()  # 양자화된 값을 실수형 텐서로 변환하여 모델에 저장
    weights_abs.append(quantized)  # 양자화된 가중치를 리스트에 추가
# - `absmax_quantize` 함수는 절대 최대값을 기준으로 입력 데이터를 양자화합니다.
# - `quantized.float()`는 양자화된 정수 범위의 값을 실수형 텐서로 변환하여, 모델의 파라미터 데이터에 다시 할당합니다.
# - Absmax 방식은 0을 중심으로 scaling되기 때문에, 이번 단순 모델에서는 전체 분포 구조가 비교적 유지될 수 있습니다.
# - `weights_abs` 리스트에 양자화된 가중치들을 저장하여 나중에 비교하거나 분석할 수 있도록 합니다.

# 제로포인트 양자화를 적용할 모델 생성
###### [실습] 마찬가지로 `deepcopy`를 사용하여 원본 모델을 복사합니다. ######
model_zp = deepcopy(model)
# - 이 복사본(`model_zp`)에 제로포인트 기반 양자화를 적용할 것입니다.

# 모든 가중치에 제로포인트 양자화 적용
weights_zp = []
for param in model_zp.parameters():
    ###### [실습] zeropoint_quantize 함수로 입력 텐서(`param.data`)를 제로포인트 양자화하세요. ######
    quantized, dequantized, _, _ = zeropoint_quantize(param.data)
    param.data = quantized.float()  # 양자화된 값을 실수형 텐서로 변환하여 모델에 저장
    weights_zp.append(quantized)    # 양자화된 가중치를 리스트에 추가
# - `zeropoint_quantize` 함수는 양자화된 값(`quantized`)과 복원된 값(`dequantized`)을 함께 반환합니다.
# - 여기서는 의도적으로 `quantized.float()`를 모델의 파라미터 데이터에 할당하여, dequantization 없이 사용할 때의 성능 변화를 확인합니다.
# - Zero-point 방식은 offset이 포함되어 있으므로, quantized 값을 그대로 사용하면 원래 weight 분포와 크게 달라질 수 있습니다.


#### 4-1. 원래 모델 가중치 확인

In [16]:
print("[원래 모델 weight]")
print(weights[0])

[원래 모델 weight]
tensor([[ 0.0358,  0.0252,  0.0048,  ..., -0.0170,  0.0330,  0.0043],
        [-0.0097,  0.0234,  0.0184,  ..., -0.0226,  0.0114, -0.0145],
        [ 0.0344, -0.0194, -0.0153,  ...,  0.0312,  0.0032, -0.0275],
        ...,
        [-0.0148,  0.0077, -0.0003,  ...,  0.0335, -0.0002,  0.0372],
        [ 0.0270, -0.0170, -0.0090,  ..., -0.0092,  0.0280,  0.0009],
        [-0.0133,  0.0013, -0.0089,  ..., -0.0209, -0.0082,  0.0005]],
       device='cuda:0')


#### 4-2. absolute maximum quantization 모델 가중치 확인

In [17]:
print("[Absmax quantized weight]")
print(weights_abs[0])

[Absmax quantized weight]
tensor([[ 7.,  5.,  1.,  ..., -3.,  6.,  1.],
        [-2.,  5.,  4.,  ..., -4.,  2., -3.],
        [ 7., -4., -3.,  ...,  6.,  1., -5.],
        ...,
        [-3.,  2., -0.,  ...,  7., -0.,  7.],
        [ 5., -3., -2.,  ..., -2.,  5.,  0.],
        [-3.,  0., -2.,  ..., -4., -2.,  0.]], device='cuda:0')


#### 4-3. zero-point quantization 모델 가중치 확인

In [18]:
print("[Zero-point quantized weight]")
print(weights_zp[0])

[Zero-point quantized weight]
tensor([[47., 44., 39.,  ..., 34., 46., 39.],
        [36., 44., 43.,  ..., 32., 41., 34.],
        [47., 33., 34.,  ..., 46., 39., 31.],
        ...,
        [34., 40., 38.,  ..., 47., 38., 48.],
        [45., 34., 36.,  ..., 36., 45., 38.],
        [35., 38., 36.,  ..., 33., 36., 38.]], device='cuda:0')


#### 4-4. Absolute Maximum Quantization 모델 테스트

In [19]:
abs_acc = test_model(model_abs, test_loader, print_result=False)
abs_size = get_size_of_model(model_abs)
abs_latency, abs_inference_time_per_image = benchmark_model(model_abs, test_loader, device)
print("[Weight Quantization 결과]")
print(f"Baseline Accuracy       : {baseline_acc:.2f}%")
print(f"Absmax Weight Accuracy  : {abs_acc:.2f}%")

[Weight Quantization 결과]
Baseline Accuracy       : 97.23%
Absmax Weight Accuracy  : 97.24%


#### 4-5. Zero-point Quantization 모델 테스트

In [20]:
zp_acc = test_model(model_zp, test_loader, print_result=False)
zp_size = get_size_of_model(model_zp)
zp_latency, zp_inference_time_per_image = benchmark_model(model_zp, test_loader, device)

print("[Weight Quantization 결과]")
print(f"Baseline Accuracy       : {baseline_acc:.2f}%")
print(f"Zero-point Weight Acc   : {zp_acc:.2f}%")

[Weight Quantization 결과]
Baseline Accuracy       : 97.23%
Zero-point Weight Acc   : 41.46%


#### 4-6. Zero-point Quantization 후 복원된 모델 테스트

위의 실행에서 Zero-point Quantization의 경우 모델 성능이 많이 하락할 수 있습니다.

그 이유는 Zero-point Quantization에서 생성된 `quantized` 값에는 scale과 zero-point offset이 반영되어 있기 때문입니다.

따라서 이 값을 원래 weight처럼 그대로 사용하면 각 layer의 weight 분포가 크게 달라질 수 있습니다.

이에 대한 가장 단순한 해결은 quantized 값을 다시 dequantization한 뒤, 복원된 weight를 모델에 저장하는 것입니다.


In [21]:
# 제로포인트 양자화를 적용한 뒤 dequantization된 weight를 저장할 모델 생성
dequant_model_zp = deepcopy(model)
# - `deepcopy`를 사용하여 원본 모델을 복사한 후 `dequant_model_zp`에 저장합니다.
# - 이 복사본에는 zero-point quantization 후 dequantization된 weight를 저장합니다.

# 모델의 모든 가중치에 제로포인트 양자화와 역양자화 적용
for param in dequant_model_zp.parameters():
    ###### [실습] zeropoint_quantize 함수로 입력 텐서(`param.data`)를 제로포인트 양자화하세요. ######
    quantized, dequantized, _, _ = zeropoint_quantize(param.data)
    param.data = dequantized.float()  # 복원된 값을 실수형 텐서로 변환하여 모델 파라미터에 저장
# - `param.data = dequantized.float()`을 통해 scale과 zero-point가 반영된 복원값을 모델에 저장합니다.
# - 이를 통해 quantization error는 남기되, weight의 원래 실수값 분포는 최대한 유지합니다.

# 역양자화된 모델을 테스트 데이터로 평가
dequant_zp_acc = test_model(dequant_model_zp, test_loader, print_result=False)
dequant_zp_size = get_size_of_model(dequant_model_zp)
dequant_zp_latency, dequant_zp_inference_time_per_image = benchmark_model(dequant_model_zp, test_loader, device)

print("[Zero-point Quantization 후 복원된 모델 결과]")
print(f"Baseline Accuracy             : {baseline_acc:.2f}%")
print(f"Zero-point Raw Weight Acc     : {zp_acc:.2f}%")
print(f"Zero-point Dequant Weight Acc : {dequant_zp_acc:.2f}%")

[Zero-point Quantization 후 복원된 모델 결과]
Baseline Accuracy             : 97.23%
Zero-point Raw Weight Acc     : 41.46%
Zero-point Dequant Weight Acc : 97.21%


# 5. Activation Calibration & Quantized Inference

weight는 모델이 학습된 뒤 고정되지만, activation은 입력 데이터마다 달라집니다. 따라서 activation을 양자화하려면 먼저 calibration data를 사용해 layer별 activation range를 관찰해야 합니다.

아래에서는 `SimpleNN`의 다음 activation range를 관찰합니다.

1. 입력 이미지가 flatten 된 뒤의 range
2. `fc1 + ReLU` 이후 activation range
3. `fc2 + ReLU` 이후 activation range

그 다음 관찰된 range를 사용하여 forward 과정에서 activation을 `quantize -> dequantize`합니다.

아래 코드는 calibration data를 모델에 통과시키면서 각 activation의 min/max range를 저장합니다.  

이때 `input`, `act1`, `act2`라는 이름을 사용하여 layer별 activation range를 구분합니다.

In [25]:
def update_range(range_dict, name, x):
    # activation의 최솟값과 최댓값을 업데이트하는 함수
    x_min = x.detach().min().cpu()
    x_max = x.detach().max().cpu()

    if name not in range_dict:
        range_dict[name] = {'min': x_min, 'max': x_max}
    else:
        range_dict[name]['min'] = torch.minimum(range_dict[name]['min'], x_min)
        range_dict[name]['max'] = torch.maximum(range_dict[name]['max'], x_max)


def collect_activation_ranges(model, data_loader, num_batches=10):
    # calibration data를 사용하여 activation range를 수집하는 함수
    model.eval()
    ranges = {}

    with torch.no_grad():
        for i, (images, _) in enumerate(data_loader):
            if i >= num_batches:
                break

            images = images.to(device)
            x = images.view(-1, 28 * 28)
            ###### [실습] flatten된 입력 activation의 range를 `'input'` 이름으로 업데이트하세요. ######
            update_range(ranges, 'input', x)

            x = model.relu(model.fc1(x))
            ###### [실습] 첫 번째 layer 이후 activation range를 `'act1'` 이름으로 업데이트하세요. ######
            update_range(ranges, 'act1', x)

            x = model.relu(model.fc2(x))
            ###### [실습] 두 번째 layer 이후 activation range를 `'act2'` 이름으로 업데이트하세요. ######
            update_range(ranges, 'act2', x)

    return ranges

activation_ranges = collect_activation_ranges(model, train_loader, num_batches=10)

print("[Calibration으로 얻은 activation range]")
for name, r in activation_ranges.items():
    print(f"{name:>8s} | min: {r['min']:.4f} | max: {r['max']:.4f}")

[Calibration으로 얻은 activation range]
   input | min: -0.4242 | max: 2.8215
    act1 | min: 0.0000 | max: 25.6958
    act2 | min: 0.0000 | max: 22.9338


In [26]:
class ActivationQuantizedSimpleNN(SimpleNN):
    def __init__(self, activation_ranges):
        super(ActivationQuantizedSimpleNN, self).__init__()
        self.activation_ranges = activation_ranges

    def _quant_dequant_activation(self, x, name):
        # calibration에서 얻은 range를 사용하여 activation을 quantize -> dequantize 합니다.
        r = self.activation_ranges[name]
        ###### [실습] calibration에서 얻은 `r['min']`과 `r['max']`를 사용하여 activation을 quantize-dequantize 하세요. ######
        _, x_dequant, _, _ = quantize_dequantize_asymmetric(x, r['min'], r['max'])
        return x_dequant

    def forward(self, x):
        # 입력 이미지를 1차원 벡터로 변환
        x = x.view(-1, 28 * 28)

        # 입력 activation quantization
        x = self._quant_dequant_activation(x, 'input')

        # 첫 번째 레이어
        x = self.relu(self.fc1(x))
        x = self._quant_dequant_activation(x, 'act1')

        # 두 번째 레이어
        x = self.relu(self.fc2(x))
        x = self._quant_dequant_activation(x, 'act2')

        # 출력 레이어
        x = self.fc3(x)
        return x

# Activation quantization만 적용한 모델
###### [실습] `ActivationQuantizedSimpleNN` 클래스와 calibration range를 사용하여 activation quantized model을 생성하세요. ######
model_act = ActivationQuantizedSimpleNN(activation_ranges).to(device)
model_act.load_state_dict(model.state_dict())

# Weight + Activation quantization을 함께 적용한 모델
model_w_act = ActivationQuantizedSimpleNN(activation_ranges).to(device)
model_w_act.load_state_dict(model_zp.state_dict())

<All keys matched successfully>

In [27]:
# Activation quantization 결과 평가
act_acc = test_model(model_act, test_loader, print_result=False)
w_act_acc = test_model(model_w_act, test_loader, print_result=False)

act_size = get_size_of_model(model_act)
w_act_size = get_size_of_model(model_w_act)

act_latency, act_inference_time_per_image = benchmark_model(model_act, test_loader, device)
w_act_latency, w_act_inference_time_per_image = benchmark_model(model_w_act, test_loader, device)

print("[Activation Quantization 결과]")
print(f"Baseline Accuracy                      : {baseline_acc:.2f}%")
print(f"Activation Quantization Acc            : {act_acc:.2f}%")
print(f"Zero-point Raw Weight + Activation Acc : {w_act_acc:.2f}%")

[Activation Quantization 결과]
Baseline Accuracy                      : 97.23%
Activation Quantization Acc            : 97.22%
Zero-point Raw Weight + Activation Acc : 28.79%


# 6. 결과 분석

아래 표는 실습에서 측정한 결과를 한 번에 비교합니다.

주의할 점은 다음과 같습니다.

- 이 노트북은 실제 INT8 backend를 사용하는 배포용 quantized model을 만드는 것이 아니라, quantization 계산 흐름과 성능 변화를 직접 확인하기 위한 실습입니다.
- 먼저 quantized weight 값을 그대로 모델에 넣었을 때 정확도가 어떻게 변하는지 확인합니다.
- Absmax 방식은 이번 단순 모델에서는 정확도가 거의 유지될 수 있지만, Zero-point 방식은 offset이 포함되어 있어 quantized 값을 그대로 weight처럼 사용하면 성능이 크게 떨어질 수 있습니다.
- Zero-point 방식은 scale과 zero-point를 이용해 dequantization한 뒤 사용할 때 원래 모델 수준의 정확도를 회복할 수 있습니다.
- 현재 모델의 `state_dict`는 여전히 FP32 tensor 형태로 저장되므로, model size가 줄어들지 않는 것이 정상입니다.
- 실제 model size와 latency 개선은 다음 실습의 PyTorch Quantization API에서 확인합니다.

In [28]:
print("Model".ljust(35), "Accuracy (%)".center(15), "Size (KB)".center(15), "Latency (ms)".center(15))
print("=" * 85)
print("Baseline FP32".ljust(35), f"{baseline_acc:.2f}".center(15), f"{baseline_size:.2f}".center(15), f"{baseline_latency * 1000:.4f}".center(15))
print("Absmax Weight Quant".ljust(35), f"{abs_acc:.2f}".center(15), f"{abs_size:.2f}".center(15), f"{abs_latency * 1000:.4f}".center(15))
print("Zero-point Weight Quant".ljust(35), f"{zp_acc:.2f}".center(15), f"{zp_size:.2f}".center(15), f"{zp_latency * 1000:.4f}".center(15))
print("Activation Quant".ljust(35), f"{act_acc:.2f}".center(15), f"{act_size:.2f}".center(15), f"{act_latency * 1000:.4f}".center(15))
print("Raw ZP Weight + Activation Quant".ljust(35), f"{w_act_acc:.2f}".center(15), f"{w_act_size:.2f}".center(15), f"{w_act_latency * 1000:.4f}".center(15))

Model                                 Accuracy (%)     Size (KB)      Latency (ms) 
Baseline FP32                            97.23           438.79          0.4143    
Absmax Weight Quant                      97.24           438.79          0.4088    
Zero-point Weight Quant                  41.46           438.79          0.3007    
Activation Quant                         97.22           438.79          1.5492    
Raw ZP Weight + Activation Quant         28.79           438.79          2.3578    


## 6-1. 결론

1. Weight quantization
    - 학습된 weight 값을 낮은 정밀도로 표현합니다.
    - Absmax 방식은 0을 중심으로 하는 대칭적인 양자화 방식입니다.
    - Zero-point 방식은 scale과 zero-point offset을 함께 사용하는 비대칭 양자화 방식입니다.
    - 따라서 Zero-point quantized 값을 그대로 weight처럼 사용하면 성능이 크게 떨어질 수 있습니다.

2. Zero-point Quantization 복원
    - Zero-point 방식에서는 quantized 값만 그대로 사용하는 것이 아니라, scale과 zero-point를 이용한 dequantization 과정이 필요합니다.
    - dequantization된 weight를 사용하면 원래 weight 분포에 가까워지고, 성능이 original 모델 수준으로 회복될 수 있습니다.

3. Activation quantization
    - 입력 데이터마다 activation 값이 달라지므로 calibration이 필요합니다.
    - calibration data가 실제 입력 분포를 잘 대표하지 못하면 정확도가 떨어질 수 있습니다.

4. 실제 배포 관점
    - 이 노트북은 quantization의 계산 흐름과 오차가 모델 정확도에 미치는 영향을 확인하기 위한 직접 구현 실습입니다.
    - 실제 model size와 latency 개선은 PyTorch quantization API 또는 하드웨어 지원 INT8 kernel을 사용할 때 확인할 수 있습니다.
